[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/ITNG/ModelingNeuralDynamics/blob/main/brian/chapter08.ipynb)

In [ ]:
import subprocess
import sys
if "google.colab" in sys.modules:
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "modelingneuraldynamics"])


# Chapter 8
## Quadratic Integrate and Fire (QIF) and Theta Neurons
- Code by : [Abolfazl Ziaeemehr](https://github.com/Ziaeemehr)

QIF_INFINITE_THRESHOLD and THREE_CIRCLES aren't simulations (a closed-form
plot and a geometric fixed-point diagram, respectively), so there's nothing
for Brian2 to add over the Python port for those two.

In [ ]:
import brian2 as b2
import matplotlib.pyplot as plt
import numpy as np

### The Quadratic Integrate-and-Fire (QIF) Neuron

Dimensionless model (threshold at v=1, reset to v=0), same convention as
chapter 7's LIF port.

In [ ]:
def simulate_QIF_neuron(tau_m, I, simulation_time, dt=0.01 * b2.ms):
    eqs = "dv/dt = (-v/tau_m*(1-v) + I)/ms : 1"
    neuron = b2.NeuronGroup(1, eqs, threshold="v>1", reset="v=0",
                             method="rk4", dt=dt,
                             namespace={"tau_m": tau_m, "I": I})
    neuron.v = 0
    st_mon = b2.StateMonitor(neuron, "v", record=True)
    sp_mon = b2.SpikeMonitor(neuron)
    net = b2.Network(neuron)
    net.add(st_mon, sp_mon)
    net.run(simulation_time)
    return st_mon, sp_mon

### Figure 8.1
QIF Voltage Trace

In [ ]:
sm1, spm1 = simulate_QIF_neuron(tau_m=2.0, I=0.15, simulation_time=150 * b2.ms)
t1 = sm1.t / b2.ms
v1 = sm1.v[0]

fig, ax = plt.subplots(figsize=(10, 5))
ax.plot(t1, v1, lw=2, c="k")
ax.set_xlim(0, 150)
ax.set_ylim(0, 2)
ax.set_xlabel("t")
ax.set_ylabel("v")
plt.tight_layout()
plt.show()

### The Theta Neuron

A continuous phase variable, no threshold/reset needed -- $1-\cos(\theta)$
sweeps from 0 up through 2 and back on its own as $\theta$ circles around.

In [ ]:
def simulate_theta_neuron(tau_m, I, simulation_time, dt=0.001 * b2.ms):
    eqs = "dtheta/dt = (-cos(theta)/tau_m + 2*I*(1+cos(theta)))/ms : 1"
    neuron = b2.NeuronGroup(1, eqs, method="rk4", dt=dt,
                             namespace={"tau_m": tau_m, "I": I})
    neuron.theta = 0
    st_mon = b2.StateMonitor(neuron, "theta", record=True)
    net = b2.Network(neuron)
    net.add(st_mon)
    net.run(simulation_time)
    return st_mon

### Figure 8.2
Theta Neuron Firing

In [ ]:
sm2 = simulate_theta_neuron(tau_m=0.5, I=0.505, simulation_time=150 * b2.ms)
t2 = sm2.t / b2.ms
theta2 = sm2.theta[0]

fig, ax = plt.subplots(figsize=(10, 5))
ax.plot(t2, 1 - np.cos(theta2), lw=2, c="k")
ax.set_xlim(0, 150)
ax.set_ylim(0, 2)
ax.set_xlabel("t")
ax.set_ylabel(r"$1-\cos(\theta)$")
ax.grid()
plt.tight_layout()
plt.show()